# Assignment 4
## Big Data Analytics

---
Question

Analyzing the Twitter Data. For this purpose follow the following steps: 
- a. Create developer’s account on twitter. One account is sufficient across 8 to 10 students.
- b. Scrap the twitter data using tweepy library. It must be done using five keywords of your choice.
- c. Use a word embedding such as GloVe and FastText to convert each tweet into vector form
- d. Perform the kmeans clustering using cosine similarity measure

---
This Notebook is dedicated to the scrapping of the twitter data only.

## Scrapping Twitter Data
- Using Tweepy library to scrap the twitter data. (only 100 req per month allowed on free tier)
- The data is scrapped using the following keywords:
    - "Python"
    - "Big Data"
    - "Natural Language Processing"
    - "Compiler Design"
    - "System Design"
- The data is saved in a CSV file for further processing (on kaggel).

### Installing the required libraries

In [3]:
%pip install tweepy

Note: you may need to restart the kernel to use updated packages.Defaulting to user installation because normal site-packages is not writeable




[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Libraries and Set Up Authentication

In [2]:
import time
import pandas as pd
from datetime import datetime
import os
import tweepy
import json
import random
import csv


# Loading the json having the credentials
twitter = json.load(open('C:\\Users\\debat\\OneDrive\\Documents\\private\\TWEET.json', 'r', encoding='utf-8'))

In [5]:
# Assigning the values to the respective variables
API_KEY = twitter['TWITTER_CONSUMER_KEY']
API_SECRET = twitter['TWITTER_CONSUMER_SECRET']
ACCESS_TOKEN = twitter['TWITTER_ACCESS_TOKEN']
ACCESS_SECRET = twitter['TWITTER_ACCESS_SECRET']
BEARER_TOKEN = twitter['TWITTER_BEARER_TOKEN']

# Authenticate using OAuth 2.0 Bearer Token
try:
    client = tweepy.Client(bearer_token=BEARER_TOKEN, consumer_key=API_KEY, consumer_secret=API_SECRET, access_token=ACCESS_TOKEN, access_token_secret=ACCESS_SECRET, wait_on_rate_limit=False)
    print('Authenticated successfully.')
except Exception as e:
    print('Error during authentication:', e)

Authenticated successfully.


### Search and Collect Tweets Using Five Keywords

In [6]:
MAX_RETRIES = 3

def fetch_tweets(keyword, tweet_count):
    retries = 0
    while retries < MAX_RETRIES:
        try:
            print(f"Fetching tweets for keyword: {keyword}...")
            response = client.search_recent_tweets(
                query=keyword,
                max_results=tweet_count,
                tweet_fields=[
                    'id', 'created_at', 'text', 'author_id', 'conversation_id', 'in_reply_to_user_id',
                    'lang', 'possibly_sensitive', 'referenced_tweets', 'source', 'public_metrics'
                ],
                user_fields=[
                    'id', 'name', 'username', 'created_at', 'protected', 'verified'
                ],
                media_fields=[
                    'media_key', 'type', 'url', 'duration_ms', 'width', 'height'
                ],
                place_fields=[
                    'full_name', 'country', 'geo'
                ],
                poll_fields=[
                    'id', 'options', 'voting_status'
                ],
                expansions=[
                    'author_id', 'referenced_tweets.id', 'attachments.media_keys',
                    'attachments.poll_ids', 'geo.place_id'
                ]
            )
            return response

        except tweepy.TooManyRequests:
            print("Rate limit exceeded. Sleeping for 900 seconds...")
            time.sleep(900)

        except Exception as e:
            print(f"Error fetching tweets for keyword {keyword}: {e}")
            retries += 1
            wait_time = (2 ** retries) + random.uniform(0, 1)  # Exponential backoff
            print(f"Retrying in {wait_time:.2f} seconds...")
            time.sleep(wait_time)
    
    print(f"❌ Failed to fetch tweets for keyword {keyword} after {MAX_RETRIES} attempts.")
    return None


In [ ]:
# Define keywords and configuration
keywords = ['Python', 'Big Data', 'Compiler Design', 'Natural Language Processing', 'System Design']
tweet_count = 10
num_iterations = 2
output_file = 'twitter_data/tweets2.csv'

# Create directory for output files if not present
os.makedirs('twitter_data', exist_ok=True)

# Open file in append mode
with open(output_file, mode='a', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=[
        'tweet_id', 'created_at', 'text', 'author_id', 'conversation_id', 'in_reply_to_user_id',
        'lang', 'possibly_sensitive', 'source', 'like_count', 'retweet_count',
        'reply_count', 'quote_count', 'username', 'user_name', 'user_created_at',
        'user_protected', 'user_verified', 'media_type', 'media_url', 'media_duration_ms',
        'media_width', 'media_height', 'place_name', 'place_country', 'place_latitude',
        'place_longitude', 'poll_duration_minutes', 'poll_end_time', 'poll_status', 'poll_options'
    ])
    
    # Write header only if the file is empty
    if file.tell() == 0:
        writer.writeheader()

    for iteration in range(1, num_iterations + 1):
        print(f"\n🚀 Starting Iteration {iteration}...")
        
        for keyword in keywords:
            response = fetch_tweets(keyword, tweet_count)
            if response and response.data:
                for tweet in response.data:
                    # Basic tweet data
                    tweet_data = {
                        'tweet_id': tweet.id,
                        'created_at': tweet.created_at,
                        'text': tweet.text,
                        'author_id': tweet.author_id,
                        'conversation_id': tweet.conversation_id if tweet.conversation_id else None,
                        'in_reply_to_user_id': tweet.in_reply_to_user_id if tweet.in_reply_to_user_id else None,
                        'lang': tweet.lang if tweet.lang else None,
                        'possibly_sensitive': tweet.possibly_sensitive if tweet.possibly_sensitive else None,
                        'source': tweet.source if tweet.source else None,
                        'like_count': tweet.public_metrics.get('like_count', 0),
                        'retweet_count': tweet.public_metrics.get('retweet_count', 0),
                        'reply_count': tweet.public_metrics.get('reply_count', 0),
                        'quote_count': tweet.public_metrics.get('quote_count', 0),
                    }

                    # User info (if included)
                    user = next((u for u in response.includes.get('users', []) if u.id == tweet.author_id), None)
                    if user:
                        tweet_data.update({
                            'username': user.username,
                            'user_name': user.name,
                            'user_created_at': user.created_at,
                            'user_protected': user.protected,
                            'user_verified': user.verified,
                        })

                    # Media info (if attached)
                    media_keys = tweet.attachments.get('media_keys', []) if tweet.attachments else []
                    for media_key in media_keys:
                        media = next((m for m in response.includes.get('media', []) if m.media_key == media_key), None)
                        if media:
                            tweet_data.update({
                                'media_type': media.type,
                                'media_url': getattr(media, 'url', None),
                                'media_duration_ms': getattr(media, 'duration_ms', None),
                                'media_width': getattr(media, 'width', None),
                                'media_height': getattr(media, 'height', None),
                            })

                    # Place info (if attached)
                    if tweet.geo:
                        place = next((p for p in response.includes.get('places', []) if p.id == tweet.geo.get('place_id')), None)
                        if place:
                            tweet_data.update({
                                'place_name': place.full_name,
                                'place_country': place.country,
                                'place_latitude': place.geo['geometry']['coordinates'][1],
                                'place_longitude': place.geo['geometry']['coordinates'][0]
                            })

                    # Poll info (if attached)
                    poll_ids = tweet.attachments.get('poll_ids', []) if tweet.attachments else []
                    for poll_id in poll_ids:
                        poll = next((p for p in response.includes.get('polls', []) if p.id == poll_id), None)
                        if poll:
                            tweet_data.update({
                                'poll_duration_minutes': poll.duration_minutes,
                                'poll_end_time': poll.end_datetime,
                                'poll_status': poll.voting_status,
                                'poll_options': '; '.join([f"{opt.label} ({opt.get('votes', 0)})" for opt in poll.options])
                            })

                    # Write data directly to file
                    writer.writerow(tweet_data)
                    file.flush()  # Ensure data is written to disk

            time.sleep(30)  # Respect Twitter's rate limit
        
        print(f"✅ Completed Iteration {iteration}\n")

print("🎯 All iterations completed successfully!")


🚀 Starting Iteration 1...
Fetching tweets for keyword: Python...
Rate limit exceeded. Sleeping for 900 seconds...


In [2]:
import pandas as pd
df = pd.read_csv('./twitter_data/tweets.csv')
df.head()

,tweet_id,created_at,text,author_id,conversation_id,in_reply_to_user_id,lang,possibly_sensitive,source,like_count,...,media_width,media_height,place_name,place_country,place_latitude,place_longitude,poll_duration_minutes,poll_end_time,poll_status,poll_options
0,1900449867664626115,2025-03-14 07:32:11+00:00,@vibeofswetha @AshokDharbal @rishibagree Flexi...,1816315585363599361,1900063037458927631,1.738046e+18,en,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1900449820789133743,2025-03-14 07:32:00+00:00,RT @Wat_the_deuce: Monty Python 😁 https://t.co...,1373580142765805573,1900449820789133743,NaN,cy,NaN,NaN,0,...,720.0,1024.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1900449800027271244,2025-03-14 07:31:55+00:00,RT @quantscience_: 🚨BREAKING: A new Python lib...,23241398,1900449800027271244,NaN,en,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1900449796864852058,2025-03-14 07:31:54+00:00,ストレスMAXでほぼ3日間ごろ寝のみで過ごした状態で本屋に寄っちゃダメだった🤦‍♀️\n\n...,1573286778668453888,1900449796864852058,NaN,ja,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1900449779596840988,2025-03-14 07:31:50+00:00,"Yes I am very capable, mainly on python or win...",746566256350756864,1900449779596840988,NaN,en,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df2 = pd.read_csv('./twitter_data/tweets2.csv')
df2.head()

,tweet_id,created_at,text,author_id,conversation_id,in_reply_to_user_id,lang,possibly_sensitive,source,like_count,...,media_width,media_height,place_name,place_country,place_latitude,place_longitude,poll_duration_minutes,poll_end_time,poll_status,poll_options
0,1900468334589620479,2025-03-14 08:45:34+00:00,RT @OpenAI: OpenAI o1 and o3-mini now offer Py...,1324755800590614528,1900468334589620479,NaN,en,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1900468304646754317,2025-03-14 08:45:27+00:00,ported lode's vandevenne's raycaster to pure p...,1064634643511537664,1900468304646754317,NaN,en,NaN,NaN,0,...,1920.0,1080.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1900468302809583921,2025-03-14 08:45:27+00:00,"RT @dataduck_: duckdb를 왜 mysql, postgreSQL과 비교...",104677542,1900468302809583921,NaN,ko,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1900468294148120587,2025-03-14 08:45:24+00:00,@kawirabrenda231 Python 🐍🐍,1433820996470034434,1900463257162322361,1.609966e+18,cy,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1900468246337175688,2025-03-14 08:45:13+00:00,📌 3ステップでできる！天気を自動取得 🚀\n\n毎朝の天気チェック、自動化しませんか？☀️...,1793199143932084224,1900468246337175688,NaN,ja,NaN,NaN,1,...,1280.0,1776.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print("Shape of df1:", df.shape)  # df is the DataFrame from cell 11
print("Shape of df2:", df2.shape)  # df2 is the DataFrame from cell 12

Shape of df1: (50, 31)
Shape of df2: (50, 31)


It means the DataFrame has 50 rows and 31 columns/features. In Each dataframe.

### Lets Merge the datasets into one csv file

In [7]:
# Combine both files into one DataFrame
df_merge = pd.concat([df, df2], ignore_index=True)

# Preview the data
print(df_merge.head())

              tweet_id                 created_at  \
0  1900449867664626115  2025-03-14 07:32:11+00:00   
1  1900449820789133743  2025-03-14 07:32:00+00:00   
2  1900449800027271244  2025-03-14 07:31:55+00:00   
3  1900449796864852058  2025-03-14 07:31:54+00:00   
4  1900449779596840988  2025-03-14 07:31:50+00:00   

                                                text            author_id  \
0  @vibeofswetha @AshokDharbal @rishibagree Flexi...  1816315585363599361   
1  RT @Wat_the_deuce: Monty Python 😁 https://t.co...  1373580142765805573   
2  RT @quantscience_: 🚨BREAKING: A new Python lib...             23241398   
3  ストレスMAXでほぼ3日間ごろ寝のみで過ごした状態で本屋に寄っちゃダメだった🤦‍♀️\n\n...  1573286778668453888   
4  Yes I am very capable, mainly on python or win...   746566256350756864   

       conversation_id  in_reply_to_user_id lang  possibly_sensitive  source  \
0  1900063037458927631         1.738046e+18   en                 NaN     NaN   
1  1900449820789133743                  NaN   cy          

In [8]:
df_merge.shape

(100, 31)

In [9]:
df_merge.to_csv('./twitter_data/all_tweets.csv', index=False)
print("✅ Merged DataFrame saved to 'merged_tweets.csv'")

✅ Merged DataFrame saved to 'merged_tweets.csv'


In [11]:
df_merge = pd.read_csv('./twitter_data/all_tweets.csv')
df_merge.shape

(100, 31)

### FINALLY I HAVE 100 tweets information to complete the assignment